# Session-based Binary Prediction with XGBoost Baseline

XGBoost counterpart to `BCE_no_pretrain_0409.ipynb`. Consumes the same session DataFrames
(sequential categorical/numerical + static categorical/numerical + binary target).

Sequences are truncated to the last `MAX_SEQ_LEN` events and **right-aligned** into
fixed-width columns `feat_0 ... feat_{MAX_SEQ_LEN-1}`, so the most recent event is always
in the last column (mirrors left-padding in the transformer). Heavy class imbalance is
handled via `scale_pos_weight`.

In [ ]:
import os
import csv
import json
import datetime

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

print(f"XGBoost version: {xgb.__version__}")

## Configuration

In [ ]:
# ---------------------------------------------------------------------------
# Feature configuration — must match the column names in train_df / test_df
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURES = [
    "sess_pid_seq",
    "sess_csid_seq",
    "sess_ccid_seq",
    "sess_bid_seq",
]

CONTINUOUS_FEATURES = [
    "sess_price_log_norm_seq",
    "sess_dtime_secs_log_norm_seq",
    "sess_prod_recency_days_log_norm_seq",
]

STATIC_CATEGORICAL_FEATURES = [
    "user_segment",
    "device_type",
]

STATIC_CONTINUOUS_FEATURES = [
    "user_lifetime_days",
    "user_avg_order_value",
]

TARGET_COLUMN = "target"

# ---------------------------------------------------------------------------
# Sequence flattening
# ---------------------------------------------------------------------------
MAX_SEQ_LEN = 20
# Fill value used for left-padded (empty) positions. Matches the user-specified
# example: for sequence [X0,X1,X2,X3] with MAX_SEQ_LEN=5, output is [0, X0, X1, X2, X3].
# Swap to np.nan if you prefer XGBoost's native missing-value routing for padded slots.
PAD_VALUE = 0

# ---------------------------------------------------------------------------
# Training hyperparameters
# ---------------------------------------------------------------------------
NUM_BOOST_ROUND = 500
EVAL_EVERY = 10          # print + log eval metrics every N boosting rounds
EARLY_STOPPING_ROUNDS = 50
POS_WEIGHT = 20.0        # upweight positives — wired into XGB as scale_pos_weight

XGB_PARAMS = {
    "objective":        "binary:logistic",  # predict returns probabilities in [0, 1]
    "eval_metric":      "auc",
    "tree_method":      "hist",
    "max_depth":        6,
    "learning_rate":    0.05,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 1.0,
    "reg_alpha":        0.0,
    "reg_lambda":       1.0,
    "gamma":            0.0,
    "scale_pos_weight": POS_WEIGHT,
}

COMMENT = ""  # free-text notes for this training run

print(f"Target: {TARGET_COLUMN}")
print(f"Sequential features: {len(CATEGORICAL_FEATURES)} cat + {len(CONTINUOUS_FEATURES)} cont")
print(f"Static features:     {len(STATIC_CATEGORICAL_FEATURES)} cat + {len(STATIC_CONTINUOUS_FEATURES)} cont")
print(f"MAX_SEQ_LEN={MAX_SEQ_LEN}, PAD_VALUE={PAD_VALUE}, POS_WEIGHT={POS_WEIGHT}")

## Load Data

Provide `train_df` and `test_df` as pandas DataFrames. Each row is one session with
list-valued sequence columns, scalar static columns, and a binary `target` column.

In [ ]:
# train_df = pd.read_parquet("train.parquet")
# test_df  = pd.read_parquet("test.parquet")

print(f"Train: {len(train_df)} sessions")
print(f"Test:  {len(test_df)} sessions")
print(f"Train positive rate: {train_df[TARGET_COLUMN].mean():.4%}")
print(f"Test  positive rate: {test_df[TARGET_COLUMN].mean():.4%}")

## Sequence Flattening

Truncate each sequence to its last `MAX_SEQ_LEN` events, then right-align into
`feat_0 ... feat_{MAX_SEQ_LEN-1}`. A sequence of length 4 with `MAX_SEQ_LEN=5`
becomes `[PAD, X0, X1, X2, X3]` — the most recent event is always in `feat_{MAX_SEQ_LEN-1}`.

In [ ]:
def _flatten_seq_column(series: pd.Series, dtype) -> np.ndarray:
    """Right-align variable-length sequences into a (N, MAX_SEQ_LEN) array."""
    n = len(series)
    out = np.full((n, MAX_SEQ_LEN), PAD_VALUE, dtype=dtype)
    for i, seq in enumerate(series.values):
        arr = np.asarray(seq, dtype=dtype)[-MAX_SEQ_LEN:]
        k = len(arr)
        if k:
            out[i, MAX_SEQ_LEN - k:] = arr
    return out


def flatten_sessions(df: pd.DataFrame) -> pd.DataFrame:
    """Expand sequential features into fixed-width columns and concat static features.

    Categorical columns are kept as plain int64 here — the `category` dtype is applied
    later (after the unseen-value sanity check) so train and test share the same
    category set.
    """
    frames = []

    # --- Sequential categorical features (int) ---
    for feat in CATEGORICAL_FEATURES:
        mat = _flatten_seq_column(df[feat], dtype=np.int64)
        cols = [f"{feat}_{i}" for i in range(MAX_SEQ_LEN)]
        frames.append(pd.DataFrame(mat, columns=cols, index=df.index))

    # --- Sequential continuous features (float) ---
    for feat in CONTINUOUS_FEATURES:
        mat = _flatten_seq_column(df[feat], dtype=np.float32)
        cols = [f"{feat}_{i}" for i in range(MAX_SEQ_LEN)]
        frames.append(pd.DataFrame(mat, columns=cols, index=df.index))

    # --- Static categorical features (scalar int) ---
    if STATIC_CATEGORICAL_FEATURES:
        frames.append(df[STATIC_CATEGORICAL_FEATURES].astype(np.int64))

    # --- Static continuous features (scalar float) ---
    if STATIC_CONTINUOUS_FEATURES:
        frames.append(df[STATIC_CONTINUOUS_FEATURES].astype(np.float32))

    return pd.concat(frames, axis=1)


def _seq_cat_columns(feat: str):
    return [f"{feat}_{i}" for i in range(MAX_SEQ_LEN)]


def all_categorical_columns():
    """Every flattened column that should be treated as categorical by XGBoost."""
    cols = []
    for feat in CATEGORICAL_FEATURES:
        cols.extend(_seq_cat_columns(feat))
    cols.extend(STATIC_CATEGORICAL_FEATURES)
    return cols


# Sanity check on the first train row
_wide = flatten_sessions(train_df.head(2))
print(f"Flattened columns: {_wide.shape[1]}")
print(f"Expected:          {MAX_SEQ_LEN * (len(CATEGORICAL_FEATURES) + len(CONTINUOUS_FEATURES)) + len(STATIC_CATEGORICAL_FEATURES) + len(STATIC_CONTINUOUS_FEATURES)}")
print(f"\nSample flattened columns (first seq feature):")
print([c for c in _wide.columns if c.startswith(CATEGORICAL_FEATURES[0])])
del _wide

## Build Train / Test Matrices

In [ ]:
X_train = flatten_sessions(train_df)
y_train = train_df[TARGET_COLUMN].astype(np.float32).values

X_test = flatten_sessions(test_df)
y_test = test_df[TARGET_COLUMN].astype(np.float32).values

print(f"X_train: {X_train.shape}, y_train positives: {int(y_train.sum())}/{len(y_train)} ({y_train.mean():.4%})")
print(f"X_test:  {X_test.shape}, y_test positives:  {int(y_test.sum())}/{len(y_test)} ({y_test.mean():.4%})")

## Unseen-Category Sanity Check

For each categorical feature, find values present in test but never seen in train,
report per-feature counts of distinct unseen values and their total frequency, then
remap every unseen test value to `UNSEEN_FILL` (= `PAD_VALUE = 0`). After remapping,
both splits are cast to the **same** `CategoricalDtype` so XGBoost sees a consistent
category set across train and test.

In [ ]:
UNSEEN_FILL = PAD_VALUE  # unseen-in-train values in test are rewritten to this

unseen_report = {}

def _remap_unseen(train_values: np.ndarray, test_block: np.ndarray):
    """Return (remapped_test_block, stats_dict) where unseen values are set to UNSEEN_FILL."""
    seen = np.unique(train_values)
    flat = test_block.ravel()
    unseen_mask = ~np.isin(flat, seen, assume_unique=False)
    n_occ = int(unseen_mask.sum())
    total = int(flat.size)
    if n_occ:
        unseen_vals = flat[unseen_mask]
        vals, counts = np.unique(unseen_vals, return_counts=True)
        order = np.argsort(-counts)
        top = [(int(vals[j]), int(counts[j])) for j in order[:10]]
        flat = flat.copy()
        flat[unseen_mask] = UNSEEN_FILL
        remapped = flat.reshape(test_block.shape)
    else:
        vals = np.array([], dtype=flat.dtype)
        top = []
        remapped = test_block
    stats = {
        "n_distinct_unseen": int(len(vals)),
        "n_occurrences_unseen": n_occ,
        "test_total": total,
        "unseen_rate": (n_occ / total) if total else 0.0,
        "top_unseen": top,
    }
    return remapped, stats


print(f"{'feature':<28} {'distinct_unseen':>16} {'occ_unseen':>12} {'test_total':>12} {'rate':>8}")
print("-" * 80)

# --- Sequential categorical features: aggregate seen-set across all MAX_SEQ_LEN positions ---
for feat in CATEGORICAL_FEATURES:
    cols = _seq_cat_columns(feat)
    train_block = X_train[cols].to_numpy()
    test_block = X_test[cols].to_numpy()
    remapped, stats = _remap_unseen(train_block, test_block)
    if stats["n_occurrences_unseen"]:
        X_test[cols] = remapped
    unseen_report[feat] = stats
    print(f"{feat:<28} {stats['n_distinct_unseen']:>16,} {stats['n_occurrences_unseen']:>12,} "
          f"{stats['test_total']:>12,} {stats['unseen_rate']:>7.2%}")
    if stats["top_unseen"]:
        print(f"  top unseen (value, count): {stats['top_unseen']}")

# --- Static categorical features ---
for feat in STATIC_CATEGORICAL_FEATURES:
    train_block = X_train[[feat]].to_numpy()
    test_block = X_test[[feat]].to_numpy()
    remapped, stats = _remap_unseen(train_block, test_block)
    if stats["n_occurrences_unseen"]:
        X_test[feat] = remapped.ravel()
    unseen_report[feat] = stats
    print(f"{feat:<28} {stats['n_distinct_unseen']:>16,} {stats['n_occurrences_unseen']:>12,} "
          f"{stats['test_total']:>12,} {stats['unseen_rate']:>7.2%}")
    if stats["top_unseen"]:
        print(f"  top unseen (value, count): {stats['top_unseen']}")

# ---------------------------------------------------------------------------
# Cast to shared CategoricalDtype — FEATURE-LEVEL for sequential features.
#
# All MAX_SEQ_LEN position columns of the same logical feature share ONE dtype
# built from the union of train values across all positions (plus UNSEEN_FILL).
# This matches the granularity of the remap above, and has the bonus that every
# position uses the same category codes — so a tree split on "value X at any
# position" is coherent.
# ---------------------------------------------------------------------------
cat_dtypes = {}

for feat in CATEGORICAL_FEATURES:
    cols = _seq_cat_columns(feat)
    cats = sorted(set(np.unique(X_train[cols].to_numpy()).tolist()) | {UNSEEN_FILL})
    dtype = pd.CategoricalDtype(categories=cats)
    for c in cols:
        X_train[c] = X_train[c].astype(dtype)
        X_test[c]  = X_test[c].astype(dtype)
        cat_dtypes[c] = dtype

for feat in STATIC_CATEGORICAL_FEATURES:
    cats = sorted(set(X_train[feat].unique().tolist()) | {UNSEEN_FILL})
    dtype = pd.CategoricalDtype(categories=cats)
    X_train[feat] = X_train[feat].astype(dtype)
    X_test[feat]  = X_test[feat].astype(dtype)
    cat_dtypes[feat] = dtype

# Post-remap invariant: every test categorical value is now a known category.
for c in all_categorical_columns():
    n_na = int(X_test[c].isna().sum())
    assert n_na == 0, f"Unknown category leaked into {c} after remap ({n_na} NaNs)"


def prepare_inference_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten + remap unseen categoricals + cast to shared dtypes.

    Use this for any new data at inference time so its categorical encoding matches
    what the booster was trained on. Unseen-category handling is feature-level for
    sequential features (matching the training-time remap).
    """
    wide = flatten_sessions(df)

    for feat in CATEGORICAL_FEATURES:
        cols = _seq_cat_columns(feat)
        allowed = np.asarray(cat_dtypes[cols[0]].categories)
        arr = wide[cols].to_numpy()
        mask = ~np.isin(arr, allowed)
        if mask.any():
            arr = arr.copy()
            arr[mask] = UNSEEN_FILL
            wide[cols] = arr
        for c in cols:
            wide[c] = wide[c].astype(cat_dtypes[c])

    for feat in STATIC_CATEGORICAL_FEATURES:
        allowed = np.asarray(cat_dtypes[feat].categories)
        vals = wide[feat].to_numpy()
        mask = ~np.isin(vals, allowed)
        if mask.any():
            vals = vals.copy()
            vals[mask] = UNSEEN_FILL
            wide[feat] = vals
        wide[feat] = wide[feat].astype(cat_dtypes[feat])

    return wide


print("\nAll unseen test values remapped to 0; feature-level categorical dtypes aligned across train/test.")

In [ ]:
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
dtest  = xgb.DMatrix(X_test,  label=y_test,  enable_categorical=True)
print(f"dtrain: {dtrain.num_row()} rows x {dtrain.num_col()} cols")
print(f"dtest:  {dtest.num_row()} rows x {dtest.num_col()} cols")

## Train with Periodic Evaluation

In [ ]:
time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
log_path = f"training_log_xgb_{time_stamp}.csv"

evals_result = {}

print(f"Training XGBoost for up to {NUM_BOOST_ROUND} rounds (eval every {EVAL_EVERY})...\n")
booster = xgb.train(
    params=XGB_PARAMS,
    dtrain=dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=[(dtrain, "train"), (dtest, "test")],
    evals_result=evals_result,
    verbose_eval=EVAL_EVERY,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)

# Dump per-round AUCs to CSV for parity with the transformer logger.
train_auc_hist = evals_result["train"]["auc"]
test_auc_hist  = evals_result["test"]["auc"]

with open(log_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["timestamp", "round", "train_auc", "test_auc"])
    now = datetime.datetime.now().isoformat()
    for i, (ta, va) in enumerate(zip(train_auc_hist, test_auc_hist), start=1):
        w.writerow([now, i, f"{ta:.6f}", f"{va:.6f}"])

print(f"\nLog written to {log_path}")
print(f"Best iteration: {booster.best_iteration + 1}  (best test AUC: {booster.best_score:.6f})")

## Final Evaluation

With `objective="binary:logistic"`, `booster.predict` returns **probabilities in `[0, 1]`**
directly — this is the predicted conversion probability.

In [ ]:
best_ntree = booster.best_iteration + 1
probs_test = booster.predict(dtest, iteration_range=(0, best_ntree))

assert probs_test.min() >= 0.0 and probs_test.max() <= 1.0, "Expected probabilities in [0,1]"

final_auc = roc_auc_score(y_test, probs_test)
print(f"Final test AUC (best iter={best_ntree}): {final_auc:.6f}")
print(f"Predicted prob range: [{probs_test.min():.4f}, {probs_test.max():.4f}]")
print(f"Mean predicted prob:  {probs_test.mean():.4f}  (ground-truth positive rate: {y_test.mean():.4%})")

## AUC Curve

In [ ]:
rounds = list(range(1, len(train_auc_hist) + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(rounds, train_auc_hist, label="train AUC", linewidth=1.5)
ax.plot(rounds, test_auc_hist,  label="test AUC",  linewidth=1.5)
ax.axvline(best_ntree, color="gray", linestyle="--", alpha=0.7,
           label=f"best iter = {best_ntree}")
ax.set_xlabel("Boosting round")
ax.set_ylabel("AUC")
ax.set_title("XGBoost AUC vs Boosting Round")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Save Model & Hyperparameters

In [ ]:
model_path = f"xgb_model_{time_stamp}.json"
booster.save_model(model_path)
print(f"Model saved to {model_path}")

hp_record = {
    "timestamp": time_stamp,
    "comment": COMMENT,
    "model_type": "xgboost",
    "target_column": TARGET_COLUMN,
    "categorical_features": CATEGORICAL_FEATURES,
    "continuous_features": CONTINUOUS_FEATURES,
    "static_categorical_features": STATIC_CATEGORICAL_FEATURES,
    "static_continuous_features": STATIC_CONTINUOUS_FEATURES,
    "max_seq_len": MAX_SEQ_LEN,
    "pad_value": PAD_VALUE,
    "num_boost_round": NUM_BOOST_ROUND,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "pos_weight": POS_WEIGHT,
    "xgb_params": XGB_PARAMS,
    "train_sessions": int(len(train_df)),
    "test_sessions": int(len(test_df)),
    "n_features": int(X_train.shape[1]),
    "best_iteration": int(best_ntree),
    "best_test_auc": float(booster.best_score),
    "final_test_auc": float(final_auc),
}

hp_path = f"training_xgb_{time_stamp}_hp.json"
with open(hp_path, "w") as f:
    json.dump(hp_record, f, indent=2)
print(f"Hyperparameters saved to {hp_path}")

## Production-Style Inference

In [ ]:
INFERENCE_CHUNK = 4096

all_probs = []
for start in range(0, len(test_df), INFERENCE_CHUNK):
    end = min(start + INFERENCE_CHUNK, len(test_df))
    chunk_wide = prepare_inference_matrix(test_df.iloc[start:end])
    dchunk = xgb.DMatrix(chunk_wide, enable_categorical=True)
    all_probs.append(booster.predict(dchunk, iteration_range=(0, best_ntree)))

all_probs = np.concatenate(all_probs).astype(np.float32)

inference_df = test_df.copy()
inference_df["prob"] = all_probs

if TARGET_COLUMN in test_df.columns:
    gt = test_df[TARGET_COLUMN].values.astype(np.float32)
    auc = roc_auc_score(gt, all_probs)
    print(f"AUC: {auc:.6f}")
    print(f"Positive rate (ground truth): {gt.mean():.4%}")
    print(f"Positive rate (pred > 0.5):   {(all_probs >= 0.5).mean():.4%}")

print(f"\nInference DataFrame shape: {inference_df.shape}")
print(inference_df[["sess_seq_len", "prob"]].head(10))

inference_df.to_pickle("inference_results_xgb.pkl")
print("\nSaved inference results to inference_results_xgb.pkl")